# NGLab Tutorial #5: Deep Learning Models

Explore advanced neural architectures: VAE for regime detection, TCN for pattern recognition, and NSTransformers.

## Learning Objectives

1. Implement VAE for market regime clustering
2. Visualize latent space representations
3. Compare TCN vs Transformer architectures
4. Understand attention mechanisms

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. Variational Autoencoder (VAE)

VAE compresses price windows into a low-dimensional latent space for regime detection.

### The VAE Objective

$$
\mathcal{L} = \underbrace{\|x - \hat{x}\|^2}_{\text{Reconstruction}} + \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{Regularization}}
$$

Where:
- $z \sim q(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$ (encoder)
- $\hat{x} = p(x|z)$ (decoder)
- KL divergence keeps latent space organized

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=60, latent_dim=2):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """The reparameterization trick: z = μ + σ·ε"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    """VAE loss = Reconstruction + KL divergence"""
    recon = F.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence: -0.5 * Σ(1 + log(σ²) - μ² - σ²)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon + kld, recon, kld

vae = VAE(input_dim=60, latent_dim=2)
print(f"VAE parameters: {sum(p.numel() for p in vae.parameters()):,}")

In [ ]:
# Generate market regimes
def generate_regime_data(n_samples=1000, seq_len=60):
    data = []
    labels = []
    
    for _ in range(n_samples):
        regime = np.random.choice(['trend', 'range', 'volatile'])
        
        if regime == 'trend':
            # Upward trend
            seq = np.cumsum(np.random.randn(seq_len) * 0.5 + 0.3)
        elif regime == 'range':
            # Mean-reverting
            seq = np.sin(np.linspace(0, 4*np.pi, seq_len)) + np.random.randn(seq_len) * 0.3
        else:  # volatile
            # High variance
            seq = np.cumsum(np.random.randn(seq_len) * 2.0)
        
        # Normalize
        seq = (seq - seq.mean()) / (seq.std() + 1e-8)
        data.append(seq)
        labels.append(regime)
    
    return np.array(data), labels

X_train, y_train = generate_regime_data(n_samples=1000)
X_test, y_test = generate_regime_data(n_samples=300)

X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)

print(f"Training data: {X_train_t.shape}")
print(f"Regime distribution: {pd.Series(y_train).value_counts().to_dict()}")

In [ ]:
# Train VAE
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
n_epochs = 100
batch_size = 64

for epoch in range(n_epochs):
    vae.train()
    total_loss = 0
    
    # Mini-batch training
    for i in range(0, len(X_train_t), batch_size):
        batch = X_train_t[i:i+batch_size]
        
        optimizer.zero_grad()
        recon, mu, logvar = vae(batch)
        loss, recon_loss, kld_loss = vae_loss(recon, batch, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {total_loss/len(X_train_t):.4f}")

print("\n✓ VAE training complete!")

In [ ]:
# Visualize latent space
vae.eval()
with torch.no_grad():
    mu, _ = vae.encode(X_test_t)
    z = mu.numpy()

# Color by regime
regime_colors = {'trend': 'green', 'range': 'blue', 'volatile': 'red'}
colors = [regime_colors[r] for r in y_test]

plt.figure(figsize=(10, 8))
plt.scatter(z[:, 0], z[:, 1], c=colors, alpha=0.6, s=50)
plt.title('VAE Latent Space (Market Regimes)', fontsize=14, fontweight='bold')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.grid(True, alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in regime_colors.items()]
plt.legend(handles=legend_elements, title='Regime')

plt.show()

## 2. Temporal Convolutional Network (TCN)

TCN uses dilated causal convolutions to capture long-range dependencies:

```
Dilation = 1:  x--x--x--x
Dilation = 2:  x----x----x----x
Dilation = 4:  x--------x--------x
```

In [ ]:
class SimpleTCN(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=3):
        super().__init__()
        
        layers = []
        for i in range(num_layers):
            dilation = 2 ** i
            in_ch = input_dim if i == 0 else hidden_dim
            
            layers.append(nn.Conv1d(
                in_ch, hidden_dim,
                kernel_size=3,
                dilation=dilation,
                padding=dilation
            ))
            layers.append(nn.ReLU())
        
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        # x: (batch, seq, features) -> (batch, features, seq)
        x = x.transpose(1, 2)
        out = self.tcn(x)
        # Take last timestep
        out = out[:, :, -1]
        return self.fc(out)

tcn = SimpleTCN()
print(f"TCN parameters: {sum(p.numel() for p in tcn.parameters()):,}")

## Summary

In this notebook, you learned:

✅ VAE architecture and reparameterization trick  
✅ Market regime detection via latent space clustering  
✅ TCN for efficient temporal modeling  
✅ Visualizing learned representations  

## Next Steps

Continue to **Notebook #6**: Hyperparameter Optimization with DEHB!

---